Benchmarking is recording how much throughput your program gives in a set amt of time.

Idea is to optimize this throughput eventually.

You can make your work easier by having micro-benchmarks for smaller sections of your code, and understanding where the bottleneck is. You can then prioritize optimizing that section of the code for biggest improvement.

You can use the `timeit` module for this. Prefer this over `time` because it's more accurate apparently.

In [ ]:
import timeit
import time

def func1():
    print("Function 1 Executing")
    time.sleep(3)
    print("Function 1 complete")

def func2():
    print("Function 2 executing")
    time.sleep(2)
    print("Function 2 complete")

t1 = timeit.Timer("func1()", setup="from __main__ import func1")
times = t1.repeat(repeat=2, number=1)

for t in times:
    print("{} Seconds: ".format(t))

t2 = timeit.Timer("func2()", setup="from __main__ import func2")
times = t2.repeat(repeat=2, number=1)
for t in times:
    print("{} Seconds: ".format(t))

In [ ]:
# this block of code from that yt tut: https://www.youtube.com/watch?v=QE903dZWfEo

import timeit
import time
import random

# comparing execution times of random.randint() and random.random()

randint_time: float = timeit.timeit(stmt="random.randint(0,1)", setup='import random', timer=time.perf_counter, number = 1000_000) # that's the default timer, so no need to put it.

random_time: float = timeit.timeit(stmt="random.random()", setup='import random', timer=time.perf_counter, number = 1000_000) # also, if number not set, defaults to 1_000_000

if __name__ == '__main__':
    print(randint_time)
    print(random_time) # random.random() is way faster...

# NOTE: 1. timeit is strictly for smaller code snippets... (usually a small block of code / one single function). To benchmark an entire script or something big, use the 'profile' module instead.
# NOTE: 2. timeit() by default turns off Garbage collection (i've no idea how GC works so this concept is a little alien to me.. Their point is that if your test code is dependent on GC, then you have to manually turn it on in the setup code.)

In [ ]:
import random
import timeit

def list_generate_random(amount: int) -> list[float]:
    random_numbers: list[float] = []
    for i in range (amount):
        random_numbers. append(random. random())
    return random_numbers

def comp_generate_random(amount: int) -> list[float]:
    return [random. random() for i in range(amount)]

n: int = 20

list_time: float = timeit.timeit(
    stmt='list_generate_random(amount=n)',
    globals=globals()
)
# NOTE: you can choose to use globals() to take in values instead of rewriting the entire thing in setup...

print(f"list_time : {list_time}")

comp_time: float = timeit.timeit(
    stmt='comp_generate_random(amount=n)',
    globals=globals()
)

print(f"comp_time : {comp_time}")

# There's also a nice functionality to repeat the same test a set number of times... use the timeit.repeat() call

comp_time_list: list[float] = timeit.repeat(
    stmt='comp_generate_random(amount=n)',
    globals=globals(),
    repeat=5
)

print(f"comp_time_list : {comp_time_list}")

### 2. Timing Decorators

you can use decorators to make the timing functionality a little less invasive to your codebase.

In [ ]:
import random
import timeit
import time
from typing import Callable, TypeVar                                                           
                                                                                                 
T = TypeVar('T')  

def timethis(func: Callable[..., T]) -> Callable[..., T]:
    def function_timer(*args, **kwargs):
        start_time = timeit.default_timer()
        return_value = func(*args, **kwargs)
        runtime = timeit.default_timer() - start_time
        print("Function {} took {} seconds to run".format(func.__name__,
        runtime))
        return return_value
    return function_timer
    
@timethis
def long_runner():
    for x in range(3):
        sleep_time = random.choice(range(1,3))
        time.sleep(sleep_time)

if __name__ == '__main__':
 long_runner()

### 3. Timing Context Manager

Define a class that acts as a context manager for whatever code that you want to time... This is a little more flexible and non-invasive because you can time arbitrary sections of code with this... The code that you're timing need not be a single function as is the case with timing decorators....

In [ ]:
from timeit import default_timer
from urllib.request import Request, urlopen
import ssl

# timer class object which you can use everywhere to benchmark whatever code you want. Just put that particular code within the Timer context manager.
class Timer(object):
    def __init__(self, verbose=False):
        self.verbose = verbose
        self.timer = default_timer

    def __enter__(self):
        self.start = default_timer()
        return self

    def __exit__(self, *args):
        end = default_timer()
        self.elapsed_secs = end - self.start
        self.elapsed = self.elapsed_secs * 1000 # millisecs
        if self.verbose:
            print('elapsed time: %f ms' % self.elapsed)

def myFunction():
        myssl = ssl.create_default_context()
        myssl.check_hostname=False
        myssl.verify_mode=ssl.CERT_NONE
        with Timer() as t:
            req = Request('https://tutorialedge.net', headers={'User-Agent': 'Mozilla/5.0'})
            response = urlopen(req, context=myssl)

        print("Elapsed Time: {} milliseconds".format(t.elapsed))

if __name__ == '__main__':
    myFunction()